*Cleaning steps:*
1. trim()
2. deduplicate
3. validate ids 
4. rename columns
5. check date
6. handle missing values

In [0]:
from pyspark.sql import functions as F

In [0]:
df = spark.read.table('workspace.bronze.crm_cust_info')
display(df.show())
display(df.printSchema())

In [0]:
# trim() string fields
for col_name, dtype in df.dtypes:
    if dtype == 'string':
        df = df.withColumn(col_name, F.trim(F.col(col_name)))
df.printSchema()

In [0]:
# rename columns
df = (
    df.withColumnRenamed("cst_id", "id")
    .withColumnRenamed("cst_key", "key")
    .withColumnRenamed("cst_firstname", "firstname")
    .withColumnRenamed("cst_lastname", "lastname")
    .withColumnRenamed("cst_marital_status", "marital_status")
    .withColumnRenamed("cst_gndr", "gender")
    .withColumnRenamed("cst_create_date", "create_date")
)
df.printSchema()

In [0]:
# validate
df = (
    df.withColumn("gender", 
        F.when(df.gender == "M", "Male")
        .when(df.gender == "F", "Female")
        .otherwise("N/A")
    ).withColumn("marital_status", 
        F.when(df.marital_status == "S", "Single")
        .when(df.marital_status == "M", "Married")
        .otherwise("N/A")
    )
)
df.show()

In [0]:
# drop null ids
df = df.filter(df.id.isNotNull())
df.count()

In [0]:
# check
df.show()

In [0]:
# write 
df.write.mode("overwrite").format("delta").saveAsTable("workspace.silver.crm_customer_info")

## still remain issues

In [0]:
# # duplicate "id"
# duplicated_ids = df.groupBy("id").agg(F.count("*").alias("count")).filter(("count > 1") & (df.id.isNotNull()))
# l = [row['id'] for row in duplicated_ids.collect()]
# print(l)
# df.filter(df.id.isin(l)).show()

In [0]:
# # duplicate "key"
# duplicated_keys = df.groupBy("key").agg(F.count("*").alias("count")).filter(("count > 1"))
# duplicated_keys.show()

In [0]:
# print(df.select(df.id).distinct().count())
# print(df.select(df.key).distinct().count())